# 02 - Risk Metrics

Compute comprehensive risk/return metrics for each asset.

In [2]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from src.data import fetch_prices, compute_returns
from src.metrics import (
    compute_all_metrics, metrics_table, sharpe_ratio, sortino_ratio,
    max_drawdown, annualized_volatility, annualized_return, cagr,
    calmar_ratio, omega_ratio, cvar, RISK_FREE_RATE
)

In [3]:
tickers = ["AAPL", "VWCE.MI", "BTC-USD", "^GSPC"]
start_date = "2015-01-01"
end_date = "2025-01-01"

prices = fetch_prices(tickers, start=start_date, end=end_date)
returns = compute_returns(prices)

benchmark = returns["^GSPC"]
assets = {t: returns[t] for t in tickers if t != "^GSPC"}

## Full Metrics Comparison Table

All metrics computed against the S&P 500 benchmark.

In [4]:
table = metrics_table(assets, benchmark_returns=benchmark)
styled = table.style.format('{:.4f}').set_caption('Risk/Return Metrics (Benchmark: S&P 500)')
styled

,CAGR,Annualized Return,Annualized Volatility,Sharpe Ratio,Sortino Ratio,Calmar Ratio,Omega Ratio,Max Drawdown,CVaR (95%),Skewness,Kurtosis,Best Day,Worst Day,Avg Daily Return,Beta,Treynor Ratio,Information Ratio,Alpha
Asset,,,,,,,,,,,,,,,,,,
AAPL,0.2791,0.2791,0.3204,0.7461,1.0611,0.7607,1.1550,-0.3143,-0.0446,0.1247,5.1742,0.1198,-0.1286,0.0012,1.1760,0.2033,0.7684,0.1359
VWCE.MI,0.1147,0.1147,0.1644,0.4543,0.5346,0.2268,1.0977,-0.3293,-0.0265,-0.7587,8.5121,0.0767,-0.0776,0.0005,0.4661,0.1603,-0.1226,0.0338
BTC-USD,0.6181,0.6181,0.6534,0.8848,1.1994,0.7543,1.2068,-0.7663,-0.0921,-0.4074,8.5250,0.2111,-0.3717,0.0028,1.1410,0.5067,0.9165,0.4780


## Risk/Return Scatter Plot

In [5]:
fig = go.Figure()
for asset_name, asset_returns in assets.items():
    fig.add_trace(go.Scatter(
        x=[annualized_volatility(asset_returns)],
        y=[annualized_return(asset_returns)],
        mode='markers+text',
        name=asset_name,
        text=[asset_name],
        textposition='top center',
        marker=dict(size=12),
    ))

fig.add_trace(go.Scatter(
    x=[annualized_volatility(benchmark)],
    y=[annualized_return(benchmark)],
    mode='markers+text',
    name='S&P 500',
    text=['S&P 500'],
    textposition='top center',
    marker=dict(size=14, symbol='star'),
))

fig.update_layout(
    title='Risk vs Return (Annualized)',
    xaxis_title='Annualized Volatility',
    yaxis_title='Annualized Return',
    template='plotly_white',
    height=600,
)
fig.show()

## Rolling Sharpe Ratio (252-day window)

In [6]:
window = 252

fig = go.Figure()
for asset_name, asset_returns in assets.items():
    rolling_ret = asset_returns.rolling(window).apply(annualized_return, raw=False)
    rolling_vol = asset_returns.rolling(window).std() * np.sqrt(252)
    rolling_sharpe = (rolling_ret - RISK_FREE_RATE) / rolling_vol
    fig.add_trace(go.Scatter(x=rolling_sharpe.index, y=rolling_sharpe, name=asset_name, mode='lines'))

fig.update_layout(
    title=f'Rolling Sharpe Ratio ({window}-day window)',
    yaxis_title='Sharpe Ratio',
    xaxis_title='Date',
    template='plotly_white',
    height=500,
)
fig.add_hline(y=0, line_dash='dash', line_color='gray')
fig.show()

## Rolling Volatility (252-day window)

In [7]:
fig = go.Figure()
for asset_name, asset_returns in assets.items():
    rolling_vol = asset_returns.rolling(252).std() * np.sqrt(252)
    fig.add_trace(go.Scatter(x=rolling_vol.index, y=rolling_vol, name=asset_name, mode='lines'))

fig.update_layout(
    title='Rolling Annualized Volatility (252-day)',
    yaxis_title='Annualized Volatility',
    xaxis_title='Date',
    template='plotly_white',
    height=500,
)
fig.show()

## Drawdown Comparison

In [8]:
fig = go.Figure()
for asset_name, asset_returns in assets.items():
    cum = (1 + asset_returns).cumprod()
    running_max = cum.cummax()
    dd = (cum - running_max) / running_max * 100
    fig.add_trace(go.Scatter(x=dd.index, y=dd, name=asset_name, mode='lines', fill='tozeroy'))

fig.update_layout(
    title='Drawdown (%)',
    yaxis_title='Drawdown %',
    xaxis_title='Date',
    template='plotly_white',
    height=500,
)
fig.show()

## Key Ratios Bar Chart

In [9]:
ratios_data = {}
for name, ret in assets.items():
    ratios_data[name] = {
        'Sharpe': sharpe_ratio(ret),
        'Sortino': sortino_ratio(ret),
        'Calmar': calmar_ratio(ret),
        'Omega': omega_ratio(ret),
    }

ratios_df = pd.DataFrame(ratios_data)

fig = go.Figure()
for ratio_name in ratios_df.index:
    fig.add_trace(go.Bar(
        x=ratios_df.columns,
        y=ratios_df.loc[ratio_name],
        name=ratio_name,
    ))
fig.update_layout(
    title='Risk-Adjusted Return Ratios',
    yaxis_title='Ratio Value',
    barmode='group',
    template='plotly_white',
    height=500,
)
fig.show()

## QuantStats Full Report

Generates a detailed HTML report for a single asset with many more metrics.

In [10]:
import quantstats as qs

qs.reports.html(assets['AAPL'], benchmark='SPY', title='AAPL Analysis', output='../reports/aapl_report.html')
print('Report saved to ../reports/aapl_report.html')

findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f

Report saved to ../reports/aapl_report.html
